In [ ]:
from IPython.display import HTML, display

display(HTML("""
<script type="module">
  import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
  mermaid.initialize({ startOnLoad: false, theme: "neutral", securityLevel: "strict" });
  const renderMermaid = async () => {
    const nodes = [...document.querySelectorAll(".mermaid:not([data-processed])")];
    if (nodes.length) await mermaid.run({ nodes });
  };
  new MutationObserver(() => renderMermaid()).observe(document.body, { childList: true, subtree: true });
  renderMermaid();
</script>
"""))


# Notebook 13 — Hybrid production architecture

This final AgentOps notebook combines the earlier lessons into a production-oriented architecture. Instead of building one giant autonomous agent, we start with a deterministic workflow that classifies the task, selects the least autonomous reliable path, applies policy checks, and pauses for human approval before consequential actions.

The core principle is simple but easy to forget: **agents are components inside a system, not the system itself**.


## Notebook-first learning contract

This notebook is the primary lesson for this topic. The Python module is not a separate replacement for the lesson; it is the implementation layer that the notebook explains, runs, breaks, and evaluates. Work through the notebook in this order:

1. read the concept model and architecture boundary;
2. inspect the tool/state/policy contracts;
3. run the deterministic implementation;
4. trigger the deliberate failure case;
5. record evaluation, cost, latency, and safety observations; and
6. answer the architecture question before moving on.


## Deep-dive training guide — Hybrid production architecture

### Concepts to master

- agents as components inside deterministic systems
- task classification before autonomy
- policy, approval, and audit outside the model

### Implementation walkthrough

`hybrid_production_architecture.py` routes simple lookup, investigation, and high-risk cases to different architectures and converges them on policy checks.

### Deliberate failure case

Send every task directly to a team. The architecture becomes impressive-looking but less predictable and harder to operate.

### Learner exercise

Add a route for compliance-sensitive customer messaging and require communications approval even when the incident is low severity.

### What to write down

For each run, capture the chosen architecture, tool trajectory, evidence used, rejected alternatives, stop condition, estimated cost, latency, and one sentence explaining whether the architecture was the least autonomous reliable option.


## Engineering checklist for this notebook

Use this checklist as your mini design review before you call the topic complete.

| Area | Question to answer |
| --- | --- |
| Control boundary | Which decisions are made by deterministic code, and which are delegated to the model? |
| Tools | Are tool inputs typed, narrow, authorized, and auditable? |
| State | What state is carried between steps, and what should never become long-term memory? |
| Failure mode | What is the easiest way this design loops, overacts, or fabricates certainty? |
| Evaluation | Which outcome, trajectory, safety, cost, and latency signals prove the design is working? |
| Architecture choice | Why is this architecture simpler or better than the nearest alternative? |


## Architecture

<pre class="mermaid">
flowchart TD
    W["Deterministic workflow"] --> C["Classify task"]
    C --> L["Simple lookup"]
    C --> I["Investigation"]
    C --> H["High-risk case"]
    L --> D["Deterministic status/report workflow"]
    I --> A["Single bounded agent"]
    H --> T["Specialist agent team"]
    D --> P["Policy checks"]
    A --> P
    T --> P
    P --> R{"High-impact action?"}
    R -- "no" --> F["Final recommendation"]
    R -- "yes" --> U["Human approval"]
    U --> X["Approved action"]
</pre>

This is the architecture ladder from the article turned into an executable router.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo / "labs") not in sys.path:
    sys.path.insert(0, str(repo / "labs"))

from agentops_lab.hybrid_production_architecture import run_examples

plans = run_examples()
[(plan.route, plan.architecture, plan.approval_required) for plan in plans]


## Inspect the routing decisions

The router does not ask the model what architecture is fashionable. It uses operational features: whether the path is known, how ambiguous the evidence is, how risky the action is, and how large the customer impact is.

In [ ]:
for plan in plans:
    print(f"{plan.route}: {plan.architecture}")
    print(f"  reason: {plan.reason}")
    print(f"  checks: {', '.join(plan.policy_checks)}")
    print(f"  approval required: {plan.approval_required}\n")


## Design checklist

A credible production agent system should be able to answer these questions before launch:

1. What task classes are handled deterministically?
2. Which tasks justify a bounded single agent?
3. Which tasks justify a specialist team, and what measured improvement beats the overhead?
4. Which tools are read-only, which are propose-only, and which require approval?
5. What budgets stop loops before cost, latency, or risk becomes unacceptable?
6. What traces and receipts prove what happened after the run?

If those answers are missing, the system is not production-ready yet.

## Reflection

1. Where would you place LangGraph in this architecture?
2. Where would OpenAI Agents SDK be enough?
3. Where would AutoGen or CrewAI make the team easier to reason about?
4. Which decisions must stay deterministic even when agents are involved?